# Guardian remote unit tests on Google Colab

Use **Runtime → Run all** in Google Colab Free. A CPU runtime is sufficient; no GPU, pip installs, credentials, or API keys are required. Internet access is needed to clone/update the public repository.

The notebook checks out `main` by default. Set `REVISION` to a branch, tag, or full commit SHA to test another revision. Existing dirty checkouts are refused to protect local work. Results describe the checked-out commit, not necessarily the notebook version.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/aminbakhtiari777/Zamith-guardian.git"
REVISION = "main"
WORKSPACE = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = WORKSPACE / "Zamith-guardian"
REPORT_PATH = WORKSPACE / "guardian_test_report.json"


def git(*args, cwd=None):
    return subprocess.check_output(
        ["git", *args], cwd=cwd, text=True, stderr=subprocess.STDOUT
    ).strip()


if not REVISION or REVISION.startswith("-"):
    raise ValueError("REVISION must be a branch, tag, or commit SHA")
if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"Refusing to reuse a non-Git directory: {REPO_DIR}")
    if git("remote", "get-url", "origin", cwd=REPO_DIR) != REPOSITORY_URL:
        raise RuntimeError("Existing checkout has an unexpected origin")
    if git("status", "--porcelain", cwd=REPO_DIR):
        raise RuntimeError("Checkout has local changes; save them before updating")
else:
    git("clone", "--", REPOSITORY_URL, str(REPO_DIR))

# FETCH_HEAD identifies exactly the requested remote revision, avoiding stale branches.
git("fetch", "origin", REVISION, cwd=REPO_DIR)
git("checkout", "--detach", "FETCH_HEAD", cwd=REPO_DIR)
COMMIT_SHA = git("rev-parse", "HEAD", cwd=REPO_DIR)
print(f"Repository: {REPO_DIR}\nRevision: {REVISION}\nCommit SHA: {COMMIT_SHA}")


In [ ]:
import json
import platform
import shutil


def detect_hardware(disk_path):
    memory = {}
    meminfo = Path("/proc/meminfo")
    if meminfo.exists():
        for line in meminfo.read_text().splitlines():
            key, value = line.split(":", 1)
            if key in {"MemTotal", "MemAvailable"}:
                memory[key] = int(value.split()[0]) * 1024
    disk = shutil.disk_usage(disk_path)
    gpus = []
    gpu_status = "No NVIDIA GPU detected; CPU tests remain available"
    if shutil.which("nvidia-smi"):
        try:
            output = subprocess.check_output([
                "nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                "--format=csv,noheader,nounits",
            ], text=True, stderr=subprocess.STDOUT, timeout=15)
            import csv
            for name, total, free in csv.reader(output.splitlines(), skipinitialspace=True):
                gpus.append({"name": name, "vram_total_bytes": int(total) * 1024**2,
                             "vram_free_bytes": int(free) * 1024**2})
            gpu_status = "NVIDIA GPU detected" if gpus else gpu_status
        except (subprocess.SubprocessError, OSError, ValueError) as exc:
            gpu_status = f"GPU information unavailable ({type(exc).__name__}); using CPU"
    return {
        "cpu": {"architecture": platform.machine(), "model": platform.processor(),
                "logical_cpus": os.cpu_count(),
                "available_logical_cpus": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()},
        "ram": {"total_bytes": memory.get("MemTotal"), "available_bytes": memory.get("MemAvailable")},
        "gpu": {"status": gpu_status, "devices": gpus},
        "disk": {"path": str(disk_path), "total_bytes": disk.total,
                 "used_bytes": disk.used, "free_bytes": disk.free},
    }


HARDWARE = detect_hardware(WORKSPACE)
print(json.dumps(HARDWARE, indent=2))


## Run every existing unit test

Tests execute in a fresh Python process using `unittest` discovery. The JSON report includes individual outcomes, failures/errors, elapsed time, hardware, Python version, and commit SHA. It is saved **before** a failing run raises an error. Zero discovered tests also fail the run. No test or policy files are changed.


In [ ]:
# This child-process script uses only the Python standard library.
TEST_RUNNER = r"""
import io
import json
import platform
import sys
import time
import unittest
from datetime import datetime, timezone
from pathlib import Path

class RecordingResult(unittest.TextTestResult):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.outcomes = []
    def addSuccess(self, test):
        super().addSuccess(test)
        self.outcomes.append({"test": test.id(), "outcome": "passed"})

started = time.perf_counter()
suite = unittest.defaultTestLoader.discover("tests", pattern="test*.py")
result = unittest.TextTestRunner(verbosity=2, resultclass=RecordingResult).run(suite)
passed_count = len(result.outcomes)
outcomes = list(result.outcomes)
for entries, outcome in [(result.failures, "failed"), (result.errors, "error")]:
    outcomes.extend({"test": test.id(), "outcome": outcome, "traceback": detail} for test, detail in entries)
outcomes.extend({"test": test.id(), "outcome": "skipped", "reason": reason} for test, reason in result.skipped)
outcomes.extend({"test": test.id(), "outcome": "expected_failure", "traceback": detail} for test, detail in result.expectedFailures)
outcomes.extend({"test": test.id(), "outcome": "unexpected_success"} for test in result.unexpectedSuccesses)
report = {
    "schema_version": 1,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit_sha": sys.argv[2],
    "python_version": platform.python_version(),
    "total_tests": result.testsRun,
    "passed_tests": passed_count,
    "failed_tests": len(result.failures),
    "errors": len(result.errors),
    "skipped_tests": len(result.skipped),
    "expected_failures": len(result.expectedFailures),
    "unexpected_successes": len(result.unexpectedSuccesses),
    "execution_time_seconds": time.perf_counter() - started,
    "successful": result.wasSuccessful() and result.testsRun > 0,
    "test_results": outcomes,
    "hardware": json.loads(sys.argv[3]),
}
Path(sys.argv[1]).write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
sys.exit(0 if report["successful"] else 1)
"""

# Remove an older report so an interrupted run cannot display stale results.
REPORT_PATH.unlink(missing_ok=True)
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR)
env["PYTHONDONTWRITEBYTECODE"] = "1"
completed = subprocess.run(
    [sys.executable, "-c", TEST_RUNNER, str(REPORT_PATH), COMMIT_SHA, json.dumps(HARDWARE)],
    cwd=REPO_DIR, env=env, text=True, capture_output=True,
)
print(completed.stdout, end="")
print(completed.stderr, end="")
if not REPORT_PATH.exists():
    raise RuntimeError(f"Test runner exited {completed.returncode} without producing a report")
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
print("\nGuardian test summary")
for key in ("total_tests", "passed_tests", "failed_tests", "errors", "skipped_tests",
            "execution_time_seconds", "git_commit_sha", "successful"):
    print(f"{key}: {report[key]}")
print(f"JSON report saved to: {REPORT_PATH}")
if completed.returncode != 0:
    raise RuntimeError("Guardian tests failed; see the output and saved JSON report")


## Download the JSON report

Run the next cell to download the report in Colab, including after a failed test run. Colab storage is temporary; download results before the runtime is recycled. Outside Colab, the cell prints the local report path.


In [ ]:
if not REPORT_PATH.exists():
    raise FileNotFoundError("Run the test cell first")
try:
    from google.colab import files
except ImportError:
    print(f"Report available at {REPORT_PATH}")
else:
    files.download(str(REPORT_PATH))
